In [4]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import text
import numpy as np
import oracledb
import sys

#CONEXIONES DESTINO

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb

un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name = "WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@', connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

# Cargar datos de referencia desde PostgreSQL
cas = pd.read_sql_query("SELECT id_cas, cod_cas FROM dim_cas WHERE cod_cas IS NOT NULL", con=connection2)
are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
estenf = pd.read_sql_query(f"SELECT id_estenf,cod_estenf FROM dim_estenf", con=connection2)
tiphab = pd.read_sql_query(f"SELECT id_tiphab,cod_tiphab FROM dim_tiphoshab", con=connection2)
esthab = pd.read_sql_query(f"SELECT id_esthab,cod_esthab FROM dim_esthoshab", con=connection2)
estreg = pd.read_sql_query("SELECT id_estreg, cod_est FROM dim_estreg", con=engine2)

# Función para limpiar columnas
def strip_columns(df):
    return df.apply(lambda x: x.str.strip() if x.dtype == "object" and x.apply(type).eq(str).all() else x)

cas = strip_columns(cas)
are = strip_columns(are)
serv = strip_columns(serv)
estenf = strip_columns(estenf)
tiphab = strip_columns(tiphab)
esthab = strip_columns(esthab)
estreg = strip_columns(estreg)

# Obtener y ajustar la fecha de inicio
# fec_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act WHERE id_proceso=13", con=connection2)
# fec_ini = fec_ini.iloc[0, 0] - timedelta(days=1)  # Restar 1 día a la fecha de inicio

# Consulta para obtener toda la data relevante desde Oracle
query = f"""
SELECT 
    p.CENASICOD,
    p.AREHOSCOD,
    p.SERVHOSCOD,
    p.ESTENFCOD,
    p.HABCOD,
    p.HABUBIDES,
    p.TIPHABCOD,
    p.ESTHABCOD,
    p.HABESTREGCOD
FROM SGSS.HMHAB10 p
"""

# Cargar todos los datos en un DataFrame
chunk = pd.read_sql_query(query, con=engine0)

chunk = chunk.rename(columns={
        'cenasicod': 'cod_cas',
        'arehoscod': 'cod_are',
        'servhoscod': 'cod_ser',
        'estenfcod': 'cod_estenf',
        'habcod': 'cod_hab',
        'habubides': 'ubi_hab',
        'tiphabcod': 'cod_tiphab',
        'esthabcod': 'cod_esthab',
        'habestregcod': 'cod_est'
    })
    
chunk = strip_columns(chunk)

chunk = pd.merge(chunk, cas, on='cod_cas', how='left').drop('cod_cas', axis=1)

chunk = pd.merge(chunk, are, on='cod_are', how="left").drop('cod_are', axis=1)

chunk = pd.merge(chunk, serv, on='cod_ser', how='left').drop('cod_ser', axis=1)

chunk = pd.merge(chunk, estenf, on='cod_estenf', how="left").drop('cod_estenf', axis=1)

chunk = pd.merge(chunk, tiphab, on='cod_tiphab', how="left").drop('cod_tiphab', axis=1)

chunk = pd.merge(chunk, esthab, on='cod_esthab', how='left').drop('cod_esthab', axis=1)

chunk = pd.merge(chunk, estreg, on='cod_est', how="left").drop('cod_est', axis=1)

# Upsert en PostgreSQL
for _, row in chunk.iterrows():
    row_dict = row.to_dict()

    def clean_data(row_dict):
        for key, value in row_dict.items():
            if isinstance(value, str):
                row_dict[key] = value.replace('\x00', '')
        return row_dict

    row_dict = clean_data(row_dict)

    # Verificar si el registro ya existe
    check_query = text("""
    SELECT 1 FROM dim_habitaciones WHERE cod_hab = :cod_hab
    """)
    result = engine2.execute(check_query, cod_hab=row_dict['cod_hab']).fetchone()

    if result:
        # Si existe, actualizar el registro
        update_query = text("""
        UPDATE dim_habitaciones
        SET id_cas=:id_cas, id_area=:id_area, id_serv=:id_serv, id_estenf=:id_estenf, ubi_hab=:ubi_hab,
            id_tiphab=:id_tiphab, id_esthab=:id_esthab, id_estreg=:id_estreg
        WHERE cod_hab = :cod_hab
        """)
        engine2.execute(update_query, **row_dict)
    else:
        # Si no existe, insertar un nuevo registro
        insert_query = text("""
        INSERT INTO dim_habitaciones (id_cas, id_area, id_serv, id_estenf, cod_hab, ubi_hab, id_tiphab, id_esthab, id_estreg)
        VALUES (:id_cas, :id_area, :id_serv, :id_estenf, :cod_hab, :ubi_hab, :id_tiphab, :id_esthab, :id_estreg)
        """)
        engine2.execute(insert_query, **row_dict)


In [5]:
now = datetime.now().strftime('%Y-%m-%d')
query2=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_proceso=13"
c2= text(query2)
connection2.execute(c2)

In [6]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()